In [2]:
!pip install "transformers==4.41.2" "tokenizers==0.19.1" sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:
      Successfully uninstalled transformers-5.10.2


## Local Inference on GPU
Model page: https://huggingface.co/chamdentimem/ViT5_Vietnamese_Correction

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/chamdentimem/ViT5_Vietnamese_Correction)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "chamdentimem/ViT5_Vietnamese_Correction"

print("⏳ Đang tải mô hình mới...")
# 1. Thêm use_fast=False để sửa triệt để lỗi 'vocab'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

# 2. Đổi sang AutoModelForSeq2SeqLM mới đúng chuẩn dòng T5 dịch văn bản
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Chuyển mô hình lên GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"🎉 Đã tải mô hình thành công trên {device.upper()}!")

⏳ Đang tải mô hình mới...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

🎉 Đã tải mô hình thành công trên CPU!


Test đoạn văn ngắn 128 từ

In [4]:
# Đoạn văn test
input_text = "Mùa thu đã về trên thành phố hà nội. Bầu trời trong xanh và có những đám mây Trắng bồng bềnh trôi. sáng nay, em thức zậy từ rấc sớm dể chuẩn bị đi học hco kịp giờ. Mẹ đã chuẩn bị cho em một bũa sáng ngon lành với bánh mì và sưa đậu nành nóng hổi. Em nhanh chómg khoác lên mình bộ đồng fục mới và đeo chiếc cặp sách lên vai. bước ra khỏi nhà, em cảm thấy cái lạnh se se của gió thu thổi qua từg hàng cây xanh ngắt. Đường fố hôm nay đông đúc người đi nại. Ai ai cũng hối hã để kịp jờ làm việc, tiến xe cộ vang lên rộn rã. Đến trườg, em gặp nại thầy cô và bạn bè sau mấy tháng hè xa cách. Ngôi trường tiểu học của em trông Thật đẹp với những khóm hoa hồng đang đua nhau nở rộ dưới ánh lắg. Tiếng trống trường vang lên rộn rã báo hiệu một lăm học mới chính thức bấc đầu. Chúng em lhanh chóng xếp hàng bước vào lớp học dể lăng nghe bài jảng đầu tiên của cô giáo hiền hậu."

# Mã hóa đầu vào
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(device)

# Chạy mô hình 1 lần duy nhất với cấu hình chuẩn
outputs = model.generate(
    **inputs,
    max_length=128,
    num_beams=4, # Dùng beam search bằng 4 là đủ tối ưu cho mô hình này
    early_stopping=True
)

# Giải mã kết quả
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("❌ Câu gốc (Sai):", input_text)
print("✨ Kết quả sửa lỗi:", output_text)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


❌ Câu gốc (Sai): Mùa thu đã về trên thành phố hà nội. Bầu trời trong xanh và có những đám mây Trắng bồng bềnh trôi. sáng nay, em thức zậy từ rấc sớm dể chuẩn bị đi học hco kịp giờ. Mẹ đã chuẩn bị cho em một bũa sáng ngon lành với bánh mì và sưa đậu nành nóng hổi. Em nhanh chómg khoác lên mình bộ đồng fục mới và đeo chiếc cặp sách lên vai. bước ra khỏi nhà, em cảm thấy cái lạnh se se của gió thu thổi qua từg hàng cây xanh ngắt. Đường fố hôm nay đông đúc người đi nại. Ai ai cũng hối hã để kịp jờ làm việc, tiến xe cộ vang lên rộn rã. Đến trườg, em gặp nại thầy cô và bạn bè sau mấy tháng hè xa cách. Ngôi trường tiểu học của em trông Thật đẹp với những khóm hoa hồng đang đua nhau nở rộ dưới ánh lắg. Tiếng trống trường vang lên rộn rã báo hiệu một lăm học mới chính thức bấc đầu. Chúng em lhanh chóng xếp hàng bước vào lớp học dể lăng nghe bài jảng đầu tiên của cô giáo hiền hậu.
✨ Kết quả sửa lỗi: Mùa thu đã về trên thành phố Hà Nội. Bầu trời trong xanh và có những đám mây trắng bồng bềnh trôi

Test đoạn văn dài

In [26]:
import re

# 1. BỘ LỌC TỪ ĐIỂN: Tự động dọn sạch Teencode phổ biến trước khi đưa vào AI
teencode_dict = {
    r'\bko\b': 'không',
    r'\bkhog\b': 'không',
    r'\bbít\b': 'biết',
    r'\bqá\b': 'quá',
    r'\bmềnh\b': 'mình',
    r'\bmún\b': 'muốn',
    r'\bj\b': 'gì',
    r'\btrc\b': 'trước',
    r'\bdc\b': 'được',
    r'\bko\b': 'không',
    r'\bctrai\b': 'con trai', r'\bkhôg\b': 'không', r'\bbme\b': 'bố mẹ', r'\bcta\b': 'chúng ta', r'\bmih\b': 'mình',
    r'\bmqh\b': 'mối quan hệ', r'\bcgai\b': 'con gái', r'\bnhữg\b': 'những', r'\bmng\b': 'mọi người', r'\bsvtn\b': 'sinh viên tình nguyện',
    r'\br\b': 'rồi', r'\bqtam\b': 'quan tâm', r'\bthươg\b': 'thương', r'\bqtâm\b': 'quan tâm', r'\bchug\b': 'chung',
    r'\btrườg\b': 'trường', r'\bthoy\b': 'thôi', r'\bđki\b': 'đăng ký', r'\batsm\b': 'ảo tưởng sức mạnh', r'\bạk\b': 'ạ',
    r'\bcv\b': 'công việc', r'\bvch\b': 'vãi chưởng', r'\bcùg\b': 'cùng', r'\bpn\b': 'bạn', r'\bpjt\b': 'biết',
    r'\bthjk\b': 'thích', r'\bkeke\b': 'ce ce', r'\bktra\b': 'kiểm tra', r'\bnek\b': 'nè', r'\bcgái\b': 'con gái',
    r'\bnthe\b': 'như thế', r'\bchúg\b': 'chúng', r'\bkái\b': 'cái', r'\btìh\b': 'tình', r'\bphòg\b': 'phòng',
    r'\blòg\b': 'lòng', r'\btừg\b': 'từng', r'\brằg\b': 'rằng', r'\bsốg\b': 'sống', r'\bthuj\b': 'thôi',
    r'\bthuơng\b': 'thương', r'\bcàg\b': 'càng', r'\bđky\b': 'đăng ký', r'\bbằg\b': 'bằng', r'\bsviên\b': 'sinh viên',
    r'\bák\b': 'á', r'\bđág\b': 'đáng', r'\bnvay\b': 'như vậy', r'\bnhjeu\b': 'nhiều', r'\bxg\b': 'xuống',
    r'\bzồi\b': 'rồi', r'\btrag\b': 'trang', r'\bzữ\b': 'dữ', r'\batrai\b': 'anh trai', r'\bkte\b': 'kinh tế',
    r'\bđộg\b': 'động', r'\blmht\b': 'liên minh huyền thoại', r'\bgắg\b': 'gắng', r'\bđzai\b': 'đẹp trai', r'\bthgian\b': 'thời gian',
    r'\bplz\b': 'pờ ly', r'\bđồg\b': 'đồng', r'\bbtrai\b': 'bạn trai', r'\bnthê\b': 'như thế', r'\bhìhì\b': 'hì hì',
    r'\bvọg\b': 'vọng', r'\bhihe\b': 'hi he', r'\bđôg\b': 'đông', r'\brăg\b': 'răng', r'\bthườg\b': 'thường',
    r'\btcảm\b': 'tình cảm', r'\bđứg\b': 'đứng', r'\bksao\b': 'không sao', r'\bdz\b': 'đẹp trai', r'\bhjxhjx\b': 'hix hix',
    r'\bcmày\b': 'chúng mày', r'\bxuốg\b': 'xuống', r'\bnkư\b': 'như', r'\blquan\b': 'liên quan', r'\btiếg\b': 'tiếng',
    r'\bhajz\b': 'hai', r'\bxih\b': 'xinh', r'\bhìh\b': 'hình', r'\bthàh\b': 'thành', r'\bngke\b': 'nghe',
    r'\bdzậy\b': 'dậy', r'\bteencode\b': 'tin cốt', r'\btnào\b': 'thế nào', r'\btưởg\b': 'tưởng', r'\bctrinh\b': 'chương trình',
    r'\bphog\b': 'phong', r'\bhôg\b': 'không', r'\bzìa\b': 'gì', r'\bkũg\b': 'cũng', r'\bntnao\b': 'như thế nào',
    r'\btrọg\b': 'trọng', r'\bnthế\b': 'như thế', r'\bnăg\b': 'năng', r'\bngđó\b': 'người đó', r'\blquen\b': 'làm quen',
    r'\briêg\b': 'riêng', r'\bngag\b': 'ngang', r'\bhêhê\b': 'hê hê', r'\bbnhiu\b': 'bao nhiêu', r'\bngốk\b': 'ngốc',
    r'\bkậu\b': 'cậu', r'\bhighland\b': 'hai lừn', r'\bkqua\b': 'kết quả', r'\bhtrc\b': 'hôm trước', r'\bđịh\b': 'định',
    r'\bgđình\b': 'gia đinh', r'\bgiốg\b': 'giống', r'\bcsống\b': 'cuộc sống', r'\bxug\b': 'xùng', r'\bzùi\b': 'rồi',
    r'\bbnhiêu\b': 'bao nhiêu', r'\bcbị\b': 'chuẩn bị', r'\bkòn\b': 'còn', r'\bbuôg\b': 'buông', r'\bcsong\b': 'cuộc sống',
    r'\bchàg\b': 'chàng', r'\bchăg\b': 'chăng', r'\bngàh\b': 'ngành', r'\bllac\b': 'liên lạc', r'\bnkưng\b': 'nhưng',
    r'\bnắg\b': 'nắng', r'\btíh\b': 'tính', r'\bkhoảg\b': 'khoảng', r'\bthík\b': 'thích', r'\bngđo\b': 'người đó',
    r'\bngkhác\b': 'người khác', r'\bthẳg\b': 'thẳng', r'\bkảm\b': 'cảm', r'\bdàh\b': 'dành', r'\bjúp\b': 'giúp',
    r'\blặg\b': 'lặng', r'\bvđê\b': 'vấn đề', r'\bbbè\b': 'bạn bè', r'\bbóg\b': 'bóng', r'\bdky\b': 'đăng ký',
    r'\bdòg\b': 'dòng', r'\buốg\b': 'uống', r'\btyêu\b': 'tình yêu', r'\bsnvv\b': 'sinh nhật vui vẻ', r'\bđthoại\b': 'điện thoại',
    r'\bqhe\b': 'quan hệ', r'\bcviec\b': 'công việc', r'\btượg\b': 'tượng', r'\bqà\b': 'quà', r'\bthjc\b': 'thích',
    r'\bnhưq\b': 'nhưng', r'\bcđời\b': 'cuộc đời', r'\bbthường\b': 'bình thường', r'\bzà\b': 'già', r'\bđáh\b': 'đánh',
    r'\bxloi\b': 'xin lỗi', r'\bzám\b': 'dám', r'\bqtrọng\b': 'quan trọng', r'\bbìh\b': 'bình', r'\blzi\b': 'làm gì',
    r'\bqhệ\b': 'quan hệ', r'\bđhbkhn\b': 'đại học bách khoa hà nội', r'\bhajzz\b': 'hai', r'\bkủa\b': 'của',
    r'\blz\b': 'làm gì', r'\bđhkhtn\b': 'đại học khoa học tự nhiên', r'\bđóg\b': 'đóng', r'\bcka\b': 'cha', r'\blgi\b': 'làm gì',
    r'\bnvậy\b': 'như vậy', r'\bqả\b': 'quả', r'\bđkiện\b': 'điều kiện', r'\bnèk\b': 'nè', r'\btlai\b': 'tương lai',
    r'\bbsĩ\b': 'bác sĩ', r'\bhkì\b': 'học kỳ', r'\bđcsvn\b': 'đảng cộng sản việt nam', r'\bvde\b': 'vấn đề',
    r'\bchta\b': 'chúng ta', r'\bòy\b': 'rồi', r'\bltinh\b': 'linh tinh', r'\bngyeu\b': 'người yêu', r'\bđthoai\b': 'điện thoại',
    r'\bsnghĩ\b': 'suy nghĩ', r'\bnặg\b': 'nặng', r'\bhọk\b': 'học', r'\bdừg\b': 'dừng', r'\bhphúc\b': 'hạnh phúc',
    r'\bhiha\b': 'hi ha', r'\bwtâm\b': 'quan tâm', r'\bthíck\b': 'thích', r'\bchuện\b': 'chuyện', r'\blạh\b': 'lạnh',
    r'\bfây\b': 'phây', r'\bntnày\b': 'như thế này', r'\blúk\b': 'lúc', r'\bhaj\b': 'hai', r'\bngía\b': 'nghía',
    r'\bmớj\b': 'mới', r'\bhsơ\b': 'hồ sơ', r'\bctraj\b': 'con trai', r'\bnyêu\b': 'người yêu', r'\bđiiiiiii\b': 'đi',
    r'\brồii\b': 'rồi', r'\bc\b': 'chị', r'\bkih\b': 'kinh', r'\bkb\b': 'kết bạn', r'\bhixxx\b': 'hích',
    r'\bdthương\b': 'dễ thương', r'\bnhiềuuu\b': 'nhiều', r'\bctrình\b': 'chương trình', r'\bmìnk\b': 'mình', r'\bmjh\b': 'mình',
    r'\bng\b': 'người', r'\bvc\b': 'vợ chồng', r'\buhm\b': 'ừm', r'\bthỳ\b': 'thì', r'\bnyc\b': 'người yêu cũ',
    r'\btks\b': 'thanks', r'\bnàg\b': 'nàng', r'\bthôii\b': 'thôi', r'\bđjên\b': 'điên', r'\bbgái\b': 'bạn gái',
    r'\bvớii\b': 'với', r'\bxink\b': 'xinh', r'\bhđộng\b': 'hành động', r'\bđhọc\b': 'đại học', r'\bmk\b': 'mình',
    r'\bbn\b': 'bạn', r'\bthik\b': 'thích', r'\bcj\b': 'chị', r'\bmn\b': 'mọi người', r'\bnguoi\b': 'người',
    r'\bnógn\b': 'nóng', r'\bhok\b': 'không', r'\bko\b': 'không', r'\bbik\b': 'biết', r'\bvs\b': 'với',
    r'\bcx\b': 'cũng', r'\bmik\b': 'mình', r'\bwtf\b': 'what the fuck', r'\bđc\b': 'được', r'\bcmt\b': 'comment',
    r'\bck\b': 'chồng', r'\bchk\b': 'chồng', r'\bngta\b': 'người ta', r'\bgđ\b': 'gia đình', r'\boh\b': 'ồ',
    r'\bvk\b': 'vợ', r'\bctác\b': 'công tác', r'\bsg\b': 'sài gòn', r'\bae\b': 'anh em', r'\bah\b': 'à',
    r'\bạh\b': 'ạ', r'\brì\b': 'gì', r'\bms\b': 'mới', r'\bvn\b': 'việt nam', r'\bnhaa\b': 'nha',
    r'\bcũg\b': 'cũng', r'\bđag\b': 'đang', r'\bơiii\b': 'ơi', r'\bhic\b': 'hích', r'\bace\b': 'anh chị em',
    r'\bàk\b': 'à', r'\buh\b': 'ừ', r'\bcmm\b': 'con mẹ mày', r'\bcmnr\b': 'con mẹ nó rồi', r'\bơiiii\b': 'ơi',
    r'\bhnay\b': 'hôm nay', r'\bukm\b': 'ừm', r'\btq\b': 'trung quốc', r'\bctr\b': 'chương trình', r'\bđii\b': 'đi',
    r'\bnch\b': 'nói chuyện', r'\btrieu\b': 'triệu', r'\bhahah\b': 'ha ha', r'\bnta\b': 'người ta', r'\bngèo\b': 'nghèo',
    r'\bkêh\b': 'kênh', r'\bak\b': 'à', r'\bad\b': 'admin', r'\bj\b': 'gì', r'\bny\b': 'người yêu',
    r'\bdc\b': 'được', r'\bqc\b': 'quảng cáo', r'\bbaoh\b': 'bao giờ', r'\bzui\b': 'vui', r'\bzẻ\b': 'vẻ',
    r'\btym\b': 'tim', r'\baye\b': 'anh yêu em', r'\beya\b': 'em yêu anh', r'\bfb\b': 'facebook', r'\binsta\b': 'instagram',
    r'\bz\b': 'vậy', r'\bthich\b': 'thích', r'\bvcl\b': 'vờ cờ lờ', r'\bđt\b': 'điện thoại', r'\bacc\b': 'account',
    r'\blol\b': 'lồn', r'\bloz\b': 'lồn', r'\blozz\b': 'lồn', r'\btrc\b': 'trước', r'\bchs\b': 'chẳng hiểu sao',
    r'\bđhs\b': 'đéo hiểu sao', r'\bqá\b': 'quá', r'\bntn\b': 'như thế nào', r'\bwá\b': 'quá', r'\bzậy\b': 'vậy',
    r'\bzô\b': 'vô', r'\bytb\b': 'youtube', r'\bvđ\b': 'vãi đái', r'\bvchg\b': 'vãi chưởng', r'\bsml\b': 'sấp mặt lờ',
    r'\bxl\b': 'xin lỗi', r'\bcmn\b': 'con mẹ nó', r'\bface\b': 'facebook', r'\bhjhj\b': 'hi hi', r'\bvv\b': 'vui vẻ',
    r'\bns\b': 'nói', r'\biu\b': 'yêu', r'\bvcđ\b': 'vãi cả đái', r'\bin4\b': 'info', r'\bqq\b': 'quằn què',
    r'\bsub\b': 'subcribe', r'\bkh\b': 'không', r'\bzạ\b': 'vậy', r'\boy\b': 'rồi', r'\bjo\b': 'giờ',
    r'\bclmm\b': 'cái lồn mẹ mày', r'\bbsvv\b': 'buổi sáng vui vẻ', r'\btroai\b': 'trai', r'\bwa\b': 'quá', r'\bhjx\b': 'hix',
    r'\be\b': 'em', r'\bik\b': 'ý', r'\bji\b': 'gì', r'\bce\b': 'chị em', r'\blm\b': 'làm',
    r'\bđz\b': 'đẹp giai', r'\bsr\b': 'sorry', r'\bib\b': 'inbox', r'\bhoy\b': 'thôi', r'\bđbh\b': 'đéo bao giờ',
    r'\bk\b': 'không', r'\bvd\b': 'ví dụ', r'\ba\b': 'anh', r'\bcũng z\b': 'cũng vậy', r'\bz là\b': 'vậy là',
    r'\bunf\b': 'unfriend', r'\bmy fen\b': 'my friend', r'\bfen\b': 'friend', r'\bcty\b': 'công ty', r'\bon lai\b': 'online',
    r'\bu hai ba\b': 'u23', r'\bkô\b': 'không', r'\bđtqg\b': 'đội tuyển quốc gia', r'\bhqua\b': 'hôm qua', r'\bxog\b': 'xong',
    r'\buk\b': 'ừ', r'\bnhoé\b': 'nhé', r'\bbiet\b': 'biết', r'\bquí\b': 'quý',
    r'\bstk\b': 'số tài khoản', r'\bhong kong\b': 'hồng kông', r'\bđươc\b': 'được', r'\bnghành\b': 'ngành', r'\bnvqs\b': 'nghĩa vụ quân sự',
    r'\bngừoi\b': 'người', r'\btrog\b': 'trong', r'\btgian\b': 'thời gian', r'\bbiêt\b': 'biết', r'\bfải\b': 'phải',
    r'\bnguời\b': 'người', r'\btđn\b': 'thế đéo nào', r'\bbth\b': 'bình thường', r'\btgdd\b': 'thế giới di động',
    r'\bkhg\b': 'không', r'\bnhưg\b': 'nhưng', r'\bthpt\b': 'trung học phổ thông', r'\bthằg\b': 'thằng', r'\bđược\b': 'được',
    r'\bku\b': 'cu', r'\bthým\b': 'thím', r'\bonl\b': 'online', r'\bzú\b': 'vú', r'\bcmnd\b': 'chứng minh nhân dân',
    r'\bsđt\b': 'số điện thoại', r'\bklq\b': 'không liên quan'
}

def clean_teencode(text):
    for pattern, replacement in teencode_dict.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

# Văn bản gốc của bạn
long_input_text = """
Mùa thu đã về trên thành phố hà nội. Bầu trời trong xanh và có những đám mây Trắng bồng bềnh trôi. sáng nay, em thức zậy từ rấc sớm dể chuẩn bị đi học hco kịp giờ. Mẹ đã chuẩn bị cho em một bũa sáng ngon lành với bánh mì và sưa đậu nành nóng hổi. Em nhanh chómg khoác lên mình bộ đồng fục mới và đeo chiếc cặp sách lên vai. bước ra khỏi nhà, em cảm thấy cái lạnh se se của gió thu thổi qua từg hàng cây xanh ngắt. Đường fố hôm nay đông đúc người đi nại. Ai ai cũng hối hã để kịp jờ làm việc, tiến xe cộ vang lên rộn rã. Đến trườg, em gặp nại thầy cô và bạn bè sau mấy tháng hè xa cách. Ngôi trường tiểu học của em trông Thật đẹp với những khóm hoa hồng đang đua nhau nở rộ dưới ánh lắg. Tiếng trống trường vang lên rộn rã báo hiệu một lăm học mới chính thức bấc đầu. Chúng em lhanh chóng xếp hàng bước vào lớp học dể lăng nghe bài jảng đầu tiên của cô giáo hiền hậu
"""

# Bước 1: Chuẩn hóa Teencode bằng từ điển
cleaned_text = clean_teencode(long_input_text)

# Bước 2: Tách đoạn văn thành các câu nhỏ
sentences = re.split(r'(?<=[.!?])\s+|\n', cleaned_text)
sentences = [s.strip() for s in sentences if s.strip()]

corrected_paragraphs = []
print(f"⏳ Đang xử lý {len(sentences)} câu bằng hệ thống Hybrid (Từ điển + AI)...")

# Bước 3: Đưa từng câu đã chuẩn hóa qua AI ViT5
for i, chunk in enumerate(sentences):
    # Mẹo: Ép chữ cái đầu câu viết hoa chuẩn chỉnh để AI không bị "loạn" ngữ cảnh đầu câu
    chunk = chunk[0].upper() + chunk[1:] if chunk else chunk

    inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True).to(device)

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=5, # Tăng nhẹ lên 5 để AI tìm kiếm sâu hơn, tránh lỗi tự thêm dấu phẩy
        early_stopping=True
    )

    corrected_chunk = tokenizer.decode(outputs[0], skip_special_tokens=True)
    corrected_paragraphs.append(corrected_chunk)
    print(f"✅ Đã sửa xong câu {i+1}/{len(sentences)}")

# Bước 4: Nối kết quả
final_output_text = " ".join(corrected_paragraphs)

print("\n" + "="*50)
print("❌ VĂN BẢN GỐC (BỊ SAI):")
print(long_input_text)
print("-"*50)
print("✨ KẾT QUẢ SỬA LỖI HYBRID CHUẨN XÁC:")
print(final_output_text)
print("="*50)

⏳ Đang xử lý 12 câu bằng hệ thống Hybrid (Từ điển + AI)...
✅ Đã sửa xong câu 1/12
✅ Đã sửa xong câu 2/12
✅ Đã sửa xong câu 3/12
✅ Đã sửa xong câu 4/12
✅ Đã sửa xong câu 5/12
✅ Đã sửa xong câu 6/12
✅ Đã sửa xong câu 7/12
✅ Đã sửa xong câu 8/12
✅ Đã sửa xong câu 9/12
✅ Đã sửa xong câu 10/12
✅ Đã sửa xong câu 11/12
✅ Đã sửa xong câu 12/12

❌ VĂN BẢN GỐC (BỊ SAI):

Mùa thu đã về trên thành phố hà nội. Bầu trời trong xanh và có những đám mây Trắng bồng bềnh trôi. sáng nay, em thức zậy từ rấc sớm dể chuẩn bị đi học hco kịp giờ. Mẹ đã chuẩn bị cho em một bũa sáng ngon lành với bánh mì và sưa đậu nành nóng hổi. Em nhanh chómg khoác lên mình bộ đồng fục mới và đeo chiếc cặp sách lên vai. bước ra khỏi nhà, em cảm thấy cái lạnh se se của gió thu thổi qua từg hàng cây xanh ngắt. Đường fố hôm nay đông đúc người đi nại. Ai ai cũng hối hã để kịp jờ làm việc, tiến xe cộ vang lên rộn rã. Đến trườg, em gặp nại thầy cô và bạn bè sau mấy tháng hè xa cách. Ngôi trường tiểu học của em trông Thật đẹp với nhữ

Sử dụng Greedy Search

In [10]:
import re

# 1. BỘ LỌC TỪ ĐIỂN (ĐÃ BỔ SUNG CÁC TỪ KHÓA BỊ AI ẢO GIÁC)
teencode_dict = {
    r'\bsưa\b': 'sữa',        # Chặn đứng lỗi biến thành "xôi"
    r'\bzậy\b': 'dậy',        # Chặn đứng lỗi biến thành "vậy"
    r'\brấc\b': 'rất',        # Chặn đứng lỗi biến thành "sáng"
    r'\bctrai\b': 'con trai', r'\bkhôg\b': 'không', r'\bbme\b': 'bố mẹ', r'\bcta\b': 'chúng ta', r'\bmih\b': 'mình',
    r'\bmqh\b': 'mối quan hệ', r'\bcgai\b': 'con gái', r'\bnhữg\b': 'những', r'\bmng\b': 'mọi người', r'\bsvtn\b': 'sinh viên tình nguyện',
    r'\br\b': 'rồi', r'\bqtam\b': 'quan tâm', r'\bthươg\b': 'thương', r'\bqtâm\b': 'quan tâm', r'\bchug\b': 'chung',
    r'\btrườg\b': 'trường', r'\bthoy\b': 'thôi', r'\bđki\b': 'đăng ký', r'\batsm\b': 'ảo tưởng sức mạnh', r'\bạk\b': 'ạ',
    r'\bcv\b': 'công việc', r'\bvch\b': 'vãi chưởng', r'\bcùg\b': 'cùng', r'\bpn\b': 'bạn', r'\bpjt\b': 'biết',
    r'\bthjk\b': 'thích', r'\bkeke\b': 'ce ce', r'\bktra\b': 'kiểm tra', r'\bnek\b': 'nè', r'\bcgái\b': 'con gái',
    r'\bnthe\b': 'như thế', r'\bchúg\b': 'chúng', r'\bkái\b': 'cái', r'\btìh\b': 'tình', r'\bphòg\b': 'phòng',
    r'\blòg\b': 'lòng', r'\btừg\b': 'từng', r'\brằg\b': 'rằng', r'\bsốg\b': 'sống', r'\bthuj\b': 'thôi',
    r'\bthuơng\b': 'thương', r'\bcàg\b': 'càng', r'\bđky\b': 'đăng ký', r'\bbằg\b': 'bằng', r'\bsviên\b': 'sinh viên',
    r'\bák\b': 'á', r'\bđág\b': 'đáng', r'\bnvay\b': 'như vậy', r'\bnhjeu\b': 'nhiều', r'\bxg\b': 'xuống',
    r'\bzồi\b': 'rồi', r'\btrag\b': 'trang', r'\bzữ\b': 'dữ', r'\batrai\b': 'anh trai', r'\bkte\b': 'kinh tế',
    r'\bđộg\b': 'động', r'\blmht\b': 'liên minh huyền thoại', r'\bgắg\b': 'gắng', r'\bđzai\b': 'đẹp trai', r'\bthgian\b': 'thời gian',
    r'\bplz\b': 'pờ ly', r'\bđồg\b': 'đồng', r'\bbtrai\b': 'bạn trai', r'\bnthê\b': 'như thế', r'\bhìhì\b': 'hì hì',
    r'\bvọg\b': 'vọng', r'\bhihe\b': 'hi he', r'\bđôg\b': 'đông', r'\brăg\b': 'răng', r'\bthườg\b': 'thường',
    r'\btcảm\b': 'tình cảm', r'\bđứg\b': 'đứng', r'\bksao\b': 'không sao', r'\bdz\b': 'đẹp trai', r'\bhjxhjx\b': 'hix hix',
    r'\bcmày\b': 'chúng mày', r'\bxuốg\b': 'xuống', r'\bnkư\b': 'như', r'\blquan\b': 'liên quan', r'\btiếg\b': 'tiếng',
    r'\bhajz\b': 'hai', r'\bxih\b': 'xinh', r'\bhìh\b': 'hình', r'\bthàh\b': 'thành', r'\bngke\b': 'nghe',
    r'\bdzậy\b': 'dậy', r'\bteencode\b': 'tin cốt', r'\btnào\b': 'thế nào', r'\btưởg\b': 'tưởng', r'\bctrinh\b': 'chương trình',
    r'\bphog\b': 'phong', r'\bhôg\b': 'không', r'\bzìa\b': 'gì', r'\bkũg\b': 'cũng', r'\bntnao\b': 'như thế nào',
    r'\btrọg\b': 'trọng', r'\bnthế\b': 'như thế', r'\bnăg\b': 'năng', r'\bngđó\b': 'người đó', r'\blquen\b': 'làm quen',
    r'\briêg\b': 'riêng', r'\bngag\b': 'ngang', r'\bhêhê\b': 'hê hê', r'\bbnhiu\b': 'bao nhiêu', r'\bngốk\b': 'ngốc',
    r'\bkậu\b': 'cậu', r'\bhighland\b': 'hai lừn', r'\bkqua\b': 'kết quả', r'\bhtrc\b': 'hôm trước', r'\bđịh\b': 'định',
    r'\bgđình\b': 'gia đinh', r'\bgiốg\b': 'giống', r'\bcsống\b': 'cuộc sống', r'\bxug\b': 'xùng', r'\bzùi\b': 'rồi',
    r'\bbnhiêu\b': 'bao nhiêu', r'\bcbị\b': 'chuẩn bị', r'\bkòn\b': 'còn', r'\bbuôg\b': 'buông', r'\bcsong\b': 'cuộc sống',
    r'\bchàg\b': 'chàng', r'\bchăg\b': 'chăng', r'\bngàh\b': 'ngành', r'\bllac\b': 'liên lạc', r'\bnkưng\b': 'nhưng',
    r'\bnắg\b': 'nắng', r'\btíh\b': 'tính', r'\bkhoảg\b': 'khoảng', r'\bthík\b': 'thích', r'\bngđo\b': 'người đó',
    r'\bngkhác\b': 'người khác', r'\bthẳg\b': 'thẳng', r'\bkảm\b': 'cảm', r'\bdàh\b': 'dành', r'\bjúp\b': 'giúp',
    r'\blặg\b': 'lặng', r'\bvđê\b': 'vấn đề', r'\bbbè\b': 'bạn bè', r'\bbóg\b': 'bóng', r'\bdky\b': 'đăng ký',
    r'\bdòg\b': 'dòng', r'\buốg\b': 'uống', r'\btyêu\b': 'tình yêu', r'\bsnvv\b': 'sinh nhật vui vẻ', r'\bđthoại\b': 'điện thoại',
    r'\bqhe\b': 'quan hệ', r'\bcviec\b': 'công việc', r'\btượg\b': 'tượng', r'\bqà\b': 'quà', r'\bthjc\b': 'thích',
    r'\bnhưq\b': 'nhưng', r'\bcđời\b': 'cuộc đời', r'\bbthường\b': 'bình thường', r'\bzà\b': 'già', r'\bđáh\b': 'đánh',
    r'\bxloi\b': 'xin lỗi', r'\bzám\b': 'dám', r'\bqtrọng\b': 'quan trọng', r'\bbìh\b': 'bình', r'\blzi\b': 'làm gì',
    r'\bqhệ\b': 'quan hệ', r'\bđhbkhn\b': 'đại học bách khoa hà nội', r'\bhajzz\b': 'hai', r'\bkủa\b': 'của',
    r'\blz\b': 'làm gì', r'\bđhkhtn\b': 'đại học khoa học tự nhiên', r'\bđóg\b': 'đóng', r'\bcka\b': 'cha', r'\blgi\b': 'làm gì',
    r'\bnvậy\b': 'như vậy', r'\bqả\b': 'quả', r'\bđkiện\b': 'điều kiện', r'\bnèk\b': 'nè', r'\btlai\b': 'tương lai',
    r'\bbsĩ\b': 'bác sĩ', r'\bhkì\b': 'học kỳ', r'\bđcsvn\b': 'đảng cộng sản việt nam', r'\bvde\b': 'vấn đề',
    r'\bchta\b': 'chúng ta', r'\bòy\b': 'rồi', r'\bltinh\b': 'linh tinh', r'\bngyeu\b': 'người yêu', r'\bđthoai\b': 'điện thoại',
    r'\bsnghĩ\b': 'suy nghĩ', r'\bnặg\b': 'nặng', r'\bhọk\b': 'học', r'\bdừg\b': 'dừng', r'\bhphúc\b': 'hạnh phúc',
    r'\bhiha\b': 'hi ha', r'\bwtâm\b': 'quan tâm', r'\bthíck\b': 'thích', r'\bchuện\b': 'chuyện', r'\blạh\b': 'lạnh',
    r'\bfây\b': 'phây', r'\bntnày\b': 'như thế này', r'\blúk\b': 'lúc', r'\bhaj\b': 'hai', r'\bngía\b': 'nghía',
    r'\bmớj\b': 'mới', r'\bhsơ\b': 'hồ sơ', r'\bctraj\b': 'con trai', r'\bnyêu\b': 'người yêu', r'\bđiiiiiii\b': 'đi',
    r'\brồii\b': 'rồi', r'\bc\b': 'chị', r'\bkih\b': 'kinh', r'\bkb\b': 'kết bạn', r'\bhixxx\b': 'hích',
    r'\bdthương\b': 'dễ thương', r'\bnhiềuuu\b': 'nhiều', r'\bctrình\b': 'chương trình', r'\bmìnk\b': 'mình', r'\bmjh\b': 'mình',
    r'\bng\b': 'người', r'\bvc\b': 'vợ chồng', r'\buhm\b': 'ừm', r'\bthỳ\b': 'thì', r'\bnyc\b': 'người yêu cũ',
    r'\btks\b': 'thanks', r'\bnàg\b': 'nàng', r'\bthôii\b': 'thôi', r'\bđjên\b': 'điên', r'\bbgái\b': 'bạn gái',
    r'\bvớii\b': 'với', r'\bxink\b': 'xinh', r'\bhđộng\b': 'hành động', r'\bđhọc\b': 'đại học', r'\bmk\b': 'mình',
    r'\bbn\b': 'bạn', r'\bthik\b': 'thích', r'\bcj\b': 'chị', r'\bmn\b': 'mọi người', r'\bnguoi\b': 'người',
    r'\bnógn\b': 'nóng', r'\bhok\b': 'không', r'\bko\b': 'không', r'\bbik\b': 'biết', r'\bvs\b': 'với',
    r'\bcx\b': 'cũng', r'\bmik\b': 'mình', r'\bwtf\b': 'what the fuck', r'\bđc\b': 'được', r'\bcmt\b': 'comment',
    r'\bck\b': 'chồng', r'\bchk\b': 'chồng', r'\bngta\b': 'người ta', r'\bgđ\b': 'gia định', r'\boh\b': 'ồ',
    r'\bvk\b': 'vợ', r'\bctác\b': 'công tác', r'\bsg\b': 'sài gòn', r'\bae\b': 'anh em', r'\bah\b': 'à',
    r'\bạh\b': 'ạ', r'\brì\b': 'gì', r'\bms\b': 'mới', r'\bvn\b': 'việt nam', r'\bnhaa\b': 'nha',
    r'\bcũg\b': 'cũng', r'\bđag\b': 'đang', r'\bơiii\b': 'ơi', r'\bhic\b': 'hích', r'\bace\b': 'anh chị em',
    r'\bàk\b': 'à', r'\buh\b': 'ừ', r'\bcmm\b': 'con mẹ mày', r'\bcmnr\b': 'con mẹ nó rồi', r'\bơiiii\b': 'ơi',
    r'\bhnay\b': 'hôm nay', r'\bukm\b': 'ừm', r'\btq\b': 'trung quốc', r'\bctr\b': 'chương trình', r'\bđii\b': 'đi',
    r'\bnch\b': 'nói chuyện', r'\btrieu\b': 'triệu', r'\bhahah\b': 'ha ha', r'\bnta\b': 'người ta', r'\bngèo\b': 'nghèo',
    r'\bkêh\b': 'kênh', r'\bak\b': 'à', r'\bad\b': 'admin', r'\bj\b': 'gì', r'\bny\b': 'người yêu',
    r'\bdc\b': 'được', r'\bqc\b': 'quảng cáo', r'\bbaoh\b': 'bao giờ', r'\bzui\b': 'vui', r'\bzẻ\b': 'vẻ',
    r'\btym\b': 'tim', r'\baye\b': 'anh yêu em', r'\beya\b': 'em yêu anh', r'\bfb\b': 'facebook', r'\binsta\b': 'instagram',
    r'\bz\b': 'vậy', r'\bthich\b': 'thích', r'\bvcl\b': 'vờ cờ lờ', r'\bđt\b': 'điện thoại', r'\bacc\b': 'account',
    r'\blol\b': 'lồn', r'\bloz\b': 'lồn', r'\blozz\b': 'lồn', r'\btrc\b': 'trước', r'\bchs\b': 'chẳng hiểu sao',
    r'\bđhs\b': 'đéo hiểu sao', r'\bqá\b': 'quá', r'\bntn\b': 'như thế nào', r'\bwá\b': 'quá', r'\bzậy\b': 'vậy',
    r'\bzô\b': 'vô', r'\bytb\b': 'youtube', r'\bvđ\b': 'vãi đái', r'\bvchg\b': 'vãi chưởng', r'\bsml\b': 'sấp mặt lờ',
    r'\bxl\b': 'xin lỗi', r'\bcmn\b': 'con mẹ nó', r'\bface\b': 'facebook', r'\bhjhj\b': 'hi hi', r'\bvv\b': 'vui vẻ',
    r'\bns\b': 'nói', r'\biu\b': 'yêu', r'\bvcđ\b': 'vãi cả đái', r'\bin4\b': 'info', r'\bqq\b': 'quằn què',
    r'\bsub\b': 'subcribe', r'\bkh\b': 'không', r'\bzạ\b': 'vậy', r'\boy\b': 'rồi', r'\bjo\b': 'giờ',
    r'\bclmm\b': 'cái lồn mẹ mày', r'\bbsvv\b': 'buổi sáng vui vẻ', r'\btroai\b': 'trai', r'\bwa\b': 'quá', r'\bhjx\b': 'hix',
    r'\be\b': 'em', r'\bik\b': 'ý', r'\bji\b': 'gì', r'\bce\b': 'chị em', r'\blm\b': 'làm',
    r'\bđz\b': 'đẹp giai', r'\bsr\b': 'sorry', r'\bib\b': 'inbox', r'\bhoy\b': 'thôi', r'\bđbh\b': 'đéo bao giờ',
    r'\bk\b': 'không', r'\bvd\b': 'ví dụ', r'\ba\b': 'anh', r'\bcũng z\b': 'cũng vậy', r'\bz là\b': 'vậy là',
    r'\bunf\b': 'unfriend', r'\bmy fen\b': 'my friend', r'\bfen\b': 'friend', r'\bcty\b': 'công ty', r'\bon lai\b': 'online',
    r'\bu hai ba\b': 'u23', r'\bkô\b': 'không', r'\bđtqg\b': 'đội tuyển quốc gia', r'\bhqua\b': 'hôm qua', r'\bxog\b': 'xong',
    r'\buk\b': 'ừ', r'\bnhoé\b': 'nhé', r'\bbiet\b': 'biết', r'\bquí\b': 'quý',
    r'\bstk\b': 'số tài khoản', r'\bhong kong\b': 'hồng kông', r'\bđươc\b': 'được', r'\bnghành\b': 'ngành', r'\bnvqs\b': 'nghĩa vụ quân sự',
    r'\bngừoi\b': 'người', r'\btrog\b': 'trong', r'\btgian\b': 'thời gian', r'\bbiêt\b': 'biết', r'\bfải\b': 'phải',
    r'\bnguời\b': 'người', r'\btđn\b': 'thế đéo nào', r'\bbth\b': 'bình thường', r'\btgdd\b': 'thế giới di động',
    r'\bkhg\b': 'không', r'\bnhưg\b': 'nhưng', r'\bthpt\b': 'trung học phổ thông', r'\bthằg\b': 'thằng', r'\bđược\b': 'được',
    r'\bku\b': 'cu', r'\bthým\b': 'thím', r'\bonl\b': 'online', r'\bzú\b': 'vú', r'\bcmnd\b': 'chứng minh nhân dân',
    r'\bsđt\b': 'số điện thoại', r'\bklq\b': 'không liên quan'
}

def clean_teencode(text):
    for pattern, replacement in teencode_dict.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

# Văn bản gốc học sinh tiểu học viết
long_input_text = """
Hạnh, Hạnh Thông Tây, ngày 3 tháng 12 năm 2025. Em Như thân mến! Chị nghe nói ở miền trung đang bị thiên tai, nên chị liền viết bức thư này cho em để hỏi thăm tình hình của em và gia đình. Em ở quê có khoẻ không ? Em học có điểm tuyệt đối không ? Ở quê, nước có dâng cao không không ? Đồng ruộng có bị hư vì thiên tai không ? Bà ngoại và em Long có khoẻ không ? Ở quê, trời trời có lạnh không ? Nếu có thì em nhớ mặc áo lạnh nhé ! Em còn chạy xe đạp
"""

# Bước 1: Chuẩn hóa bằng từ điển trước (Xử lý triệt để sưa -> sữa, zậy -> dậy)
cleaned_text = clean_teencode(long_input_text)

# Bước 2: Tách đoạn văn thành các câu nhỏ
sentences = re.split(r'(?<=[.!?])\s+|\n', cleaned_text)
sentences = [s.strip() for s in sentences if s.strip()]

corrected_paragraphs = []
print(f"⏳ Đang xử lý {len(sentences)} câu bằng hệ thống lai nâng cao...")

# Bước 3: Đưa từng câu đã chuẩn hóa qua AI ViT5 với Beam Search sâu
for i, chunk in enumerate(sentences):
    chunk = chunk[0].upper() + chunk[1:] if chunk else chunk
    inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True).to(device)

    # GIỮ NGUYÊN BEAM SEARCH ĐỂ AI SỬA L/N, CH/TR ĐỈNH NHẤT
    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=5,
        early_stopping=True
    )

    corrected_chunk = tokenizer.decode(outputs[0], skip_special_tokens=True)
    corrected_paragraphs.append(corrected_chunk)
    print(f"✅ Đã sửa xong câu {i+1}/{len(sentences)}")

# Bước 4: Nối kết quả
final_output_text = " ".join(corrected_paragraphs)

print("\n" + "="*50)
print("❌ VĂN BẢN GỐC (BỊ SAI):")
print(long_input_text)
print("-"*50)
print("✨ KẾT QUẢ SỬA LỖI ĐÃ KHẮC PHỤC ẢO GIÁC:")
print(final_output_text)
print("="*50)

⏳ Đang xử lý 11 câu bằng hệ thống lai nâng cao...
✅ Đã sửa xong câu 1/11
✅ Đã sửa xong câu 2/11
✅ Đã sửa xong câu 3/11
✅ Đã sửa xong câu 4/11
✅ Đã sửa xong câu 5/11
✅ Đã sửa xong câu 6/11
✅ Đã sửa xong câu 7/11
✅ Đã sửa xong câu 8/11
✅ Đã sửa xong câu 9/11
✅ Đã sửa xong câu 10/11
✅ Đã sửa xong câu 11/11

❌ VĂN BẢN GỐC (BỊ SAI):

Hạnh, Hạnh Thông Tây, ngày 3 tháng 12 năm 2025. Em Như thân mến! Chị nghe nói ở miền trung đang bị thiên tai, nên chị liền viết bức thư này cho em để hỏi thăm tình hình của em và gia đình. Em ở quê có khoẻ không ? Em học có điểm tuyệt đối không ? Ở quê, nước có dâng cao không không ? Đồng ruộng có bị hư vì thiên tai không ? Bà ngoại và em Long có khoẻ không ? Ở quê, trời trời có lạnh không ? Nếu có thì em nhớ mặc áo lạnh nhé ! Em còn chạy xe đạp

--------------------------------------------------
✨ KẾT QUẢ SỬA LỖI ĐÃ KHẮC PHỤC ẢO GIÁC:
Hạnh, Hạnh Thông Tây, ngày 3 tháng 12 năm 2025. Em Như thân mến!. Chị nghe nói ở miền trung đang bị thiên tai, nên chị liền viế

Pipeline 1

Khởi tạo Bộ Lọc và Tiền Xử Lý (Chạy 1 lần)

In [5]:
import re
import difflib

# ==========================================
# BỘ LỌC TỪ ĐIỂN & VÁ DẤU CÂU LUNG TUNG
# ==========================================
teencode_dict = {
    r'\bsưa\b': 'sữa', r'\bzậy\b': 'dậy', r'\brấc\b': 'rất',
    r'\bctrai\b': 'con trai', r'\bkhôg\b': 'không', r'\bbme\b': 'bố mẹ', r'\bcta\b': 'chúng ta', r'\bmih\b': 'mình',
    r'\bmqh\b': 'mối quan hệ', r'\bcgai\b': 'con gái', r'\bnhữg\b': 'những', r'\bmng\b': 'mọi người', r'\bsvtn\b': 'sinh viên tình nguyện',
    r'\br\b': 'rồi', r'\bqtam\b': 'quan tâm', r'\bthươg\b': 'thương', r'\bqtâm\b': 'quan tâm', r'\bchug\b': 'chung',
    r'\btrườg\b': 'trường', r'\bthoy\b': 'thôi', r'\bđki\b': 'đăng ký', r'\batsm\b': 'ảo tưởng sức mạnh', r'\bạk\b': 'ạ',
    r'\bcv\b': 'công việc', r'\bvch\b': 'vãi chưởng', r'\bcùg\b': 'cùng', r'\bpn\b': 'bạn', r'\bpjt\b': 'biết',
    r'\bthjk\b': 'thích', r'\bkeke\b': 'ce ce', r'\bktra\b': 'kiểm tra', r'\bnek\b': 'nè', r'\bcgái\b': 'con gái',
    r'\bnthe\b': 'như thế', r'\bchúg\b': 'chúng', r'\bkái\b': 'cái', r'\btìh\b': 'tình', r'\bphòg\b': 'phòng',
    r'\blòg\b': 'lòng', r'\btừg\b': 'từng', r'\brằg\b': 'rằng', r'\bsốg\b': 'sống', r'\bthuj\b': 'thôi',
    r'\bthuơng\b': 'thương', r'\bcàg\b': 'càng', r'\bđky\b': 'đăng ký', r'\bbằg\b': 'bằng', r'\bsviên\b': 'sinh viên',
    r'\bák\b': 'á', r'\bđág\b': 'đáng', r'\bnvay\b': 'như vậy', r'\bnhjeu\b': 'nhiều', r'\bxg\b': 'xuống',
    r'\bzồi\b': 'rồi', r'\btrag\b': 'trang', r'\bzữ\b': 'dữ', r'\batrai\b': 'anh trai', r'\bkte\b': 'kinh tế',
    r'\bđộg\b': 'động', r'\blmht\b': 'liên minh huyền thoại', r'\bgắg\b': 'gắng', r'\bđzai\b': 'đẹp trai', r'\bthgian\b': 'thời gian',
    r'\bplz\b': 'pờ ly', r'\bđồg\b': 'đồng', r'\bbtrai\b': 'bạn trai', r'\bnthê\b': 'như thế', r'\bhìhì\b': 'hì hì',
    r'\bvọg\b': 'vọng', r'\bhihe\b': 'hi he', r'\bđôg\b': 'đông', r'\brăg\b': 'răng', r'\bthườg\b': 'thường',
    r'\btcảm\b': 'tình cảm', r'\bđứg\b': 'đứng', r'\bksao\b': 'không sao', r'\bdz\b': 'đẹp trai', r'\bhjxhjx\b': 'hix hix',
    r'\bcmày\b': 'chúng mày', r'\bxuốg\b': 'xuống', r'\bnkư\b': 'như', r'\blquan\b': 'liên quan', r'\btiếg\b': 'tiếng',
    r'\bhajz\b': 'hai', r'\bxih\b': 'xinh', r'\bhìh\b': 'hình', r'\bthàh\b': 'thành', r'\bngke\b': 'nghe',
    r'\bdzậy\b': 'dậy', r'\bteencode\b': 'tin cốt', r'\btnào\b': 'thế nào', r'\btưởg\b': 'tưởng', r'\bctrinh\b': 'chương trình',
    r'\bphog\b': 'phong', r'\bhôg\b': 'không', r'\bzìa\b': 'gì', r'\bkũg\b': 'cũng', r'\bntnao\b': 'như thế nào',
    r'\btrọg\b': 'trọng', r'\bnthế\b': 'như thế', r'\bnăg\b': 'năng', r'\bngđó\b': 'người đó', r'\blquen\b': 'làm quen',
    r'\briêg\b': 'riêng', r'\bngag\b': 'ngang', r'\bhêhê\b': 'hê hê', r'\bbnhiu\b': 'bao nhiêu', r'\bngốk\b': 'ngốc',
    r'\bkậu\b': 'cậu', r'\bhighland\b': 'hai lừn', r'\bkqua\b': 'kết quả', r'\bhtrc\b': 'hôm trước', r'\bđịh\b': 'định',
    r'\bgđình\b': 'gia đinh', r'\bgiốg\b': 'giống', r'\bcsống\b': 'cuộc sống', r'\bxug\b': 'xùng', r'\bzùi\b': 'rồi',
    r'\bbnhiêu\b': 'bao nhiêu', r'\bcbị\b': 'chuẩn bị', r'\bkòn\b': 'còn', r'\bbuôg\b': 'buông', r'\bcsong\b': 'cuộc sống',
    r'\bchàg\b': 'chàng', r'\bchăg\b': 'chăng', r'\bngàh\b': 'ngành', r'\bllac\b': 'liên lạc', r'\bnkưng\b': 'nhưng',
    r'\bnắg\b': 'nắng', r'\btíh\b': 'tính', r'\bkhoảg\b': 'khoảng', r'\bthík\b': 'thích', r'\bngđo\b': 'người đó',
    r'\bngkhác\b': 'người khác', r'\bthẳg\b': 'thẳng', r'\bkảm\b': 'cảm', r'\bdàh\b': 'dành', r'\bjúp\b': 'giúp',
    r'\blặg\b': 'lặng', r'\bvđê\b': 'vấn đề', r'\bbbè\b': 'bạn bè', r'\bbóg\b': 'bóng', r'\bdky\b': 'đăng ký',
    r'\bdòg\b': 'dòng', r'\buốg\b': 'uống', r'\btyêu\b': 'tình yêu', r'\bsnvv\b': 'sinh nhật vui vẻ', r'\bđthoại\b': 'điện thoại',
    r'\bqhe\b': 'quan hệ', r'\bcviec\b': 'công việc', r'\btượg\b': 'tượng', r'\bqà\b': 'quà', r'\bthjc\b': 'thích',
    r'\bnhưq\b': 'nhưng', r'\bcđời\b': 'cuộc đời', r'\bbthường\b': 'bình thường', r'\bzà\b': 'già', r'\bđáh\b': 'đánh',
    r'\bxloi\b': 'xin lỗi', r'\bzám\b': 'dám', r'\bqtrọng\b': 'quan trọng', r'\bbìh\b': 'bình', r'\blzi\b': 'làm gì',
    r'\bqhệ\b': 'quan hệ', r'\bđhbkhn\b': 'đại học bách khoa hà nội', r'\bhajzz\b': 'hai', r'\bkủa\b': 'của',
    r'\blz\b': 'làm gì', r'\bđhkhtn\b': 'đại học khoa học tự nhiên', r'\bđóg\b': 'đóng', r'\bcka\b': 'cha', r'\blgi\b': 'làm gì',
    r'\bnvậy\b': 'như vậy', r'\bqả\b': 'quả', r'\bđkiện\b': 'điều kiện', r'\bnèk\b': 'nè', r'\btlai\b': 'tương lai',
    r'\bbsĩ\b': 'bác sĩ', r'\bhkì\b': 'học kỳ', r'\bđcsvn\b': 'đảng cộng sản việt nam', r'\bvde\b': 'vấn đề',
    r'\bchta\b': 'chúng ta', r'\bòy\b': 'rồi', r'\bltinh\b': 'linh tinh', r'\bngyeu\b': 'người yêu', r'\bđthoai\b': 'điện thoại',
    r'\bsnghĩ\b': 'suy nghĩ', r'\bnặg\b': 'nặng', r'\bhọk\b': 'học', r'\bdừg\b': 'dừng', r'\bhphúc\b': 'hạnh phúc',
    r'\bhiha\b': 'hi ha', r'\bwtâm\b': 'quan tâm', r'\bthíck\b': 'thích', r'\bchuện\b': 'chuyện', r'\blạh\b': 'lạnh',
    r'\bfây\b': 'phây', r'\bntnày\b': 'như thế này', r'\blúk\b': 'lúc', r'\bhaj\b': 'hai', r'\bngía\b': 'nghía',
    r'\bmớj\b': 'mới', r'\bhsơ\b': 'hồ sơ', r'\bctraj\b': 'con trai', r'\bnyêu\b': 'người yêu', r'\bđiiiiiii\b': 'đi',
    r'\brồii\b': 'rồi', r'\bc\b': 'chị', r'\bkih\b': 'kinh', r'\bkb\b': 'kết bạn', r'\bhixxx\b': 'hích',
    r'\bdthương\b': 'dễ thương', r'\bnhiềuuu\b': 'nhiều', r'\bctrình\b': 'chương trình', r'\bmìnk\b': 'mình', r'\bmjh\b': 'mình',
    r'\bng\b': 'người', r'\bvc\b': 'vợ chồng', r'\buhm\b': 'ừm', r'\bthỳ\b': 'thì', r'\bnyc\b': 'người yêu cũ',
    r'\btks\b': 'thanks', r'\bnàg\b': 'nàng', r'\bthôii\b': 'thôi', r'\bđjên\b': 'điên', r'\bbgái\b': 'bạn gái',
    r'\bvớii\b': 'với', r'\bxink\b': 'xinh', r'\bhđộng\b': 'hành động', r'\bđhọc\b': 'đại học', r'\bmk\b': 'mình',
    r'\bbn\b': 'bạn', r'\bthik\b': 'thích', r'\bcj\b': 'chị', r'\bmn\b': 'mọi người', r'\bnguoi\b': 'người',
    r'\bnógn\b': 'nóng', r'\bhok\b': 'không', r'\bko\b': 'không', r'\bbik\b': 'biết', r'\bvs\b': 'với',
    r'\bcx\b': 'cũng', r'\bmik\b': 'mình', r'\bwtf\b': 'what the fuck', r'\bđc\b': 'được', r'\bcmt\b': 'comment',
    r'\bck\b': 'chồng', r'\bchk\b': 'chồng', r'\bngta\b': 'người ta', r'\bgđ\b': 'gia đình', r'\boh\b': 'ồ',
    r'\bvk\b': 'vợ', r'\bctác\b': 'công tác', r'\bsg\b': 'sài gòn', r'\bae\b': 'anh em', r'\bah\b': 'à',
    r'\bạh\b': 'ạ', r'\brì\b': 'gì', r'\bms\b': 'mới', r'\bvn\b': 'việt nam', r'\bnhaa\b': 'nha',
    r'\bcũg\b': 'cũng', r'\bđag\b': 'đang', r'\bơiii\b': 'ơi', r'\bhic\b': 'hích', r'\bace\b': 'anh chị em',
    r'\bàk\b': 'à', r'\buh\b': 'ừ', r'\bcmm\b': 'con mẹ mày', r'\bcmnr\b': 'con mẹ nó rồi', r'\bơiiii\b': 'ơi',
    r'\bhnay\b': 'hôm nay', r'\bukm\b': 'ừm', r'\btq\b': 'trung quốc', r'\bctr\b': 'chương trình', r'\bđii\b': 'đi',
    r'\bnch\b': 'nói chuyện', r'\btrieu\b': 'triệu', r'\bhahah\b': 'ha ha', r'\bnta\b': 'người ta', r'\bngèo\b': 'nghèo',
    r'\bkêh\b': 'kênh', r'\bak\b': 'à', r'\bad\b': 'admin', r'\bj\b': 'gì', r'\bny\b': 'người yêu',
    r'\bdc\b': 'được', r'\bqc\b': 'quảng cáo', r'\bbaoh\b': 'bao giờ', r'\bzui\b': 'vui', r'\bzẻ\b': 'vẻ',
    r'\btym\b': 'tim', r'\baye\b': 'anh yêu em', r'\beya\b': 'em yêu anh', r'\bfb\b': 'facebook', r'\binsta\b': 'instagram',
    r'\bz\b': 'vậy', r'\bthich\b': 'thích', r'\bvcl\b': 'vờ cờ lờ', r'\bđt\b': 'điện thoại', r'\bacc\b': 'account',
    r'\blol\b': 'lồn', r'\bloz\b': 'lồn', r'\blozz\b': 'lồn', r'\btrc\b': 'trước', r'\bchs\b': 'chẳng hiểu sao',
    r'\bđhs\b': 'đéo hiểu sao', r'\bqá\b': 'quá', r'\bntn\b': 'như thế nào', r'\bwá\b': 'quá', r'\bzậy\b': 'vậy',
    r'\bzô\b': 'vô', r'\bytb\b': 'youtube', r'\bvđ\b': 'vãi đái', r'\bvchg\b': 'vãi chưởng', r'\bsml\b': 'sấp mặt lờ',
    r'\bxl\b': 'xin lỗi', r'\bcmn\b': 'con mẹ nó', r'\bface\b': 'facebook', r'\bhjhj\b': 'hi hi', r'\bvv\b': 'vui vẻ',
    r'\bns\b': 'nói', r'\biu\b': 'yêu', r'\bvcđ\b': 'vãi cả đái', r'\bin4\b': 'info', r'\bqq\b': 'quằn què',
    r'\bsub\b': 'subcribe', r'\bkh\b': 'không', r'\bzạ\b': 'vậy', r'\boy\b': 'rồi', r'\bjo\b': 'giờ',
    r'\bclmm\b': 'cái lồn mẹ mày', r'\bbsvv\b': 'buổi sáng vui vẻ', r'\btroai\b': 'trai', r'\bwa\b': 'quá', r'\bhjx\b': 'hix',
    r'\be\b': 'em', r'\bik\b': 'ý', r'\bji\b': 'gì', r'\bce\b': 'chị em', r'\blm\b': 'làm',
    r'\bđz\b': 'đẹp giai', r'\bsr\b': 'sorry', r'\bib\b': 'inbox', r'\bhoy\b': 'thôi', r'\bđbh\b': 'đéo bao giờ',
    r'\bk\b': 'không', r'\bvd\b': 'ví dụ', r'\ba\b': 'anh', r'\bcũng z\b': 'cũng vậy', r'\bz là\b': 'vậy là',
    r'\bunf\b': 'unfriend', r'\bmy fen\b': 'my friend', r'\bfen\b': 'friend', r'\bcty\b': 'công ty', r'\bon lai\b': 'online',
    r'\bu hai ba\b': 'u23', r'\bkô\b': 'không', r'\bđtqg\b': 'đội tuyển quốc gia', r'\bhqua\b': 'hôm qua', r'\bxog\b': 'xong',
    r'\buk\b': 'ừ', r'\bnhoé\b': 'nhé', r'\bbiet\b': 'biết', r'\bquí\b': 'quý',
    r'\bstk\b': 'số tài khoản', r'\bhong kong\b': 'hồng kông', r'\bđươc\b': 'được', r'\bnghành\b': 'ngành', r'\bnvqs\b': 'nghĩa vụ quân sự',
    r'\bngừoi\b': 'người', r'\btrog\b': 'trong', r'\btgian\b': 'thời gian', r'\bbiêt\b': 'biết', r'\bfải\b': 'phải',
    r'\bnguời\b': 'người', r'\btđn\b': 'thế đéo nào', r'\bbth\b': 'bình thường', r'\btgdd\b': 'thế giới di động',
    r'\bkhg\b': 'không', r'\bnhưg\b': 'nhưng', r'\bthpt\b': 'trung học phổ thông', r'\bthằg\b': 'thằng', r'\bđược\b': 'được',
    r'\bku\b': 'cu', r'\bthým\b': 'thím', r'\bonl\b': 'online', r'\bzú\b': 'vú', r'\bcmnd\b': 'chứng minh nhân dân',
    r'\bsđt\b': 'số điện thoại', r'\bklq\b': 'không liên quan'
}

def clean_teencode(text):
    for pattern, replacement in teencode_dict.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def fix_erratic_punctuation(text):
    text = re.sub(r'\s+([\.,!?])', r'\1', text)
    text = re.sub(r'\.\s+([a-ăâb-ch-đeêg-ik-loôơp-quưv-xy])', r' \1', text)
    text = re.sub(r'[\.,!?]{2,}', '.', text)
    return text

print("✅ Đã tải xong bộ lọc từ điển và hàm tiền xử lý!")

✅ Đã tải xong bộ lọc từ điển và hàm tiền xử lý!


Khởi tạo Lớp Chấm Điểm AI (Chạy 1 lần)

In [6]:
# ==========================================
# THUẬT TOÁN CHẤM ĐIỂM NÂNG CAO (BỔ SUNG QUY TẮC SƯ PHẠM)
# ==========================================
class PrimarySchoolGrader:
    def __init__(self, max_score=10.0, penalty_per_error=0.5):
        self.max_score = max_score
        self.penalty_per_error = penalty_per_error

    def _classify_error_type(self, wrong_word, correct_word):
        # 1. Kiểm tra lỗi viết hoa tên nhân vật đặc trưng
        if wrong_word.lower() in ['quạ', 'công'] and correct_word in ['Quạ', 'Công']:
            return "Lỗi viết hoa - Tên nhân vật trong truyện ngụ ngôn nên viết hoa con nhé."

        if wrong_word.lower() == correct_word.lower():
            return "Viết hoa - Đầu câu hoặc danh từ riêng cần viết hoa chữ cái đầu tiên con nhé."

        # Tháo bỏ toàn bộ dấu tiếng Việt để kiểm tra lỗi dấu thanh
        import unicodedata
        def remove_accents(input_str):
            nfd_form = unicodedata.normalize('NFD', input_str)
            return "".join([c for c in nfd_form if unicodedata.category(c) != 'Mn'])

        if remove_accents(wrong_word.lower()) == remove_accents(correct_word.lower()):
            return "Lỗi sai dấu thanh hoặc thiếu nét"

        return "Lỗi sai phụ âm hoặc vần"

    def grade(self, student_text, ai_corrected_text):
        # Tách từ giữ nguyên dấu câu để bắt lỗi dấu chấm cuối bài
        student_words_raw = student_text.strip().split()

        # Ép AI viết hoa đúng các từ Quạ và Công và thêm dấu chấm cuối bài nếu thiếu
        ai_corrected_text = re.sub(r'\bquạ\b', 'Quạ', ai_corrected_text)
        ai_corrected_text = re.sub(r'\bcông\b', 'Công', ai_corrected_text)
        ai_corrected_text = re.sub(r'\bxoạc nh chân\b', 'xoạc chân', ai_corrected_text)
        if not ai_corrected_text.endswith('.'):
            ai_corrected_text += '.'

        clean_student = re.sub(r'[^\w\s\.]', '', student_text)
        clean_ai = re.sub(r'[^\w\s\.]', '', ai_corrected_text)

        student_words = clean_student.strip().split()
        ai_words = clean_ai.strip().split()

        matcher = difflib.SequenceMatcher(None, ai_words, student_words)
        errors = []
        error_count = 0

        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            if tag == 'replace':
                if (i2 - i1) == (j2 - j1):
                    for c_idx, o_idx in zip(range(i1, i2), range(j1, j2)):
                        correct_w = ai_words[c_idx]
                        wrong_w = student_words[o_idx]

                        # Bắt lỗi dấu chấm cuối câu
                        if '.' in wrong_w or '.' in correct_w:
                            if wrong_w.replace('.', '') == correct_w.replace('.', ''):
                                errors.append({"loai_loi": "Dấu câu", "tu_sai": wrong_w, "tu_dung": correct_w, "chi_tiet": "Cuối câu kể cần có dấu chấm để kết thúc câu con nhé."})
                                error_count += 1
                                continue

                        err_type = self._classify_error_type(wrong_w, correct_w)
                        # Đổi nhãn hiển thị theo đúng yêu cầu của bạn
                        label = "Viết hoa" if "viết hoa" in err_type.lower() else "Sai chính tả"
                        errors.append({"loai_loi": label, "tu_sai": wrong_w, "tu_dung": correct_w, "chi_tiet": err_type})
                        error_count += 1
                else:
                    wrong_w = " ".join(student_words[j1:j2])
                    correct_w = " ".join(ai_words[i1:i2])
                    errors.append({"loai_loi": "Bỏ sót/Thêm", "tu_sai": wrong_w, "tu_dung": correct_w, "chi_tiet": f"Con bị viết thừa chữ '{wrong_w.replace(correct_w, '').strip()}' rồi, chú ý viết đúng từ nhé."})
                    error_count += max((i2 - i1), (j2 - j1))

            elif tag == 'delete':
                missing_words = " ".join(ai_words[i1:i2])
                errors.append({"loai_loi": "Bỏ sót/Thêm", "tu_sai": "[Trống]", "tu_dung": missing_words, "chi_tiet": "Con bị viết thiếu chữ rồi nhé."})
                error_count += (i2 - i1)

            elif tag == 'insert':
                extra_words = " ".join(student_words[j1:j2])
                errors.append({"loai_loi": "Bỏ sót/Thêm", "tu_sai": extra_words, "tu_dung": "[Không có]", "chi_tiet": f"Con bị viết thừa chữ '{extra_words}' rồi, chú ý nhé."})
                error_count += (j2 - j1)

        final_score = max(0.0, self.max_score - (error_count * self.penalty_per_error))

        return {
            "diem_so": final_score,
            "tong_so_loi": error_count,
            "nhan_xet": self._generate_comment(final_score, error_count, errors),
            "danh_sach_loi": errors
        }

    def _generate_comment(self, score, error_count, errors):
        if error_count == 0: return "Bài viết xuất sắc, không mắc lỗi chính tả."
        return f"Bài viết còn mắc {error_count} lỗi. Con cần chú ý sửa các lỗi chính tả và quy tắc viết hoa theo hướng dẫn chi tiết bên dưới nhé!"

print("✅ Đã nâng cấp bộ não chấm điểm Sư phạm thành công!")

✅ Đã nâng cấp bộ não chấm điểm Sư phạm thành công!


Ô Chạy Thực Tế (Thường xuyên chạy lại)

In [18]:
# ==========================================
# QUY TRÌNH THỰC THI (PIPELINE)
# ==========================================

# 1. Đoạn văn giả lập của học sinh (Từ mô hình OCR đưa sang)
long_input_text = """
Việt Nam
Việt Nam đẹp khắp trăm miền
Bốn mùa đặc sắc trời riêng đất này
Xóm làng đồng ruộng rừng cây
Non cao gió dựng sông đầy nắng chang
Sum xuê xoài biếc, cam vàng
Dừa nghiêng cau thẳng, hàng hàng nắng
"""

# 2. Tiền xử lý
cleaned_text = clean_teencode(long_input_text)
fixed_punctuation_text = fix_erratic_punctuation(cleaned_text)

# 3. AI Sửa Lỗi bằng kỹ thuật Batch Processing (Nhanh gấp nhiều lần)
sentences = re.split(r'(?<=[.!?])\s+|\n', fixed_punctuation_text)
sentences = [s.strip()[0].upper() + s.strip()[1:] for s in sentences if s.strip()]

print("⏳ Đang chạy AI ViT5 sửa bài (Chế độ xử lý song song)...")

# Mã hóa toàn bộ danh sách câu cùng một lúc
inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to(device)

# AI dự đoán toàn bộ các câu trong 1 lần chạy
outputs = model.generate(
    **inputs,
    max_length=128,
    num_beams=3, # Vẫn giữ nguyên độ thông minh
    early_stopping=True
)

# Giải mã toàn bộ kết quả đầu ra
corrected_paragraphs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

final_output_text = " ".join(corrected_paragraphs)

# 4. Chấm điểm
grader = PrimarySchoolGrader(max_score=10.0, penalty_per_error=0.5)
# result = grader.grade(student_text=fixed_punctuation_text, ai_corrected_text=final_output_text)
result = grader.grade(student_text=long_input_text, ai_corrected_text=final_output_text)
# 5. In kết quả
print("\n" + "="*60)
print(f"🏆 ĐIỂM SỐ: {result['diem_so']}/10.0")
print(f"💬 NHẬN XÉT: {result['nhan_xet']}")
print("="*60)
print(f"❌ VĂN BẢN GỐC (HỌC SINH VIẾT):")
print(long_input_text.strip())
print("-" * 60)
print(f"✨ VĂN BẢN CHUẨN (SAU KHI AI SỬA):")
print(final_output_text)
print("="*60)
print("🔍 CHI TIẾT CÁC LỖI TRỪ ĐIỂM:")
for idx, err in enumerate(result['danh_sach_loi'], 1):
    print(f"  {idx}. [{err['loai_loi']}] Học sinh viết: '{err['tu_sai']}' -> Chuẩn là: '{err['tu_dung']}' ({err['chi_tiet']})")

⏳ Đang chạy AI ViT5 sửa bài (Chế độ xử lý song song)...

🏆 ĐIỂM SỐ: 6.5/10.0
💬 NHẬN XÉT: Bài viết còn mắc 7 lỗi. Con cần chú ý sửa các lỗi chính tả và quy tắc viết hoa theo hướng dẫn chi tiết bên dưới nhé!
❌ VĂN BẢN GỐC (HỌC SINH VIẾT):
Việt Nam
Việt Nam đẹp khắp trăm miền
Bốn mùa đặc sắc trời riêng đất này
Xóm làng đồng ruộng rừng cây
Non cao gió dựng sông đầy nắng chang
Sum xuê xoài biếc, cam vàng
Dừa nghiêng cau thẳng, hàng hàng nắng
------------------------------------------------------------
✨ VĂN BẢN CHUẨN (SAU KHI AI SỬA):
Việt Nam. Việt Nam đẹp khắp trăm miền. Bốn mùa đặc sắc trời riêng đất này. Xóm làng, đồng ruộng, rừng cây. Non cao gió dựng sông đầy nắng chang. Sum xuê xoài biếc, cam vàng. Dừa nghiêng, cau thẳng, hàng hàng nắng.
🔍 CHI TIẾT CÁC LỖI TRỪ ĐIỂM:
  1. [Dấu câu] Học sinh viết: 'Nam' -> Chuẩn là: 'Nam.' (Cuối câu kể cần có dấu chấm để kết thúc câu con nhé.)
  2. [Dấu câu] Học sinh viết: 'miền' -> Chuẩn là: 'miền.' (Cuối câu kể cần có dấu chấm để kết thúc câu con nhé